In [2]:
import pandas as pd
import geopandas as gpd

In [5]:
# Run once to setup points
pd.read_csv("./Stasjonsdata_bløtbunnsbasen 19.12.2025.csv").rename(
    columns={"y_coord_ny": "lat", "x_coord_ny": "lon"}
).drop_duplicates(subset=["lat", "lon"])[["lat", "lon", "DYP", "LOKALITET"]].to_csv(
    "./Stasjonsdata_blotbunnsbasen_points.csv", index=False
)

In [15]:
df = pd.read_csv("https://storage.googleapis.com/niva-geodata/MarintNaturKart/Stasjonsdata_blotbunnsbasen_points.csv")
gdf_blotbunn = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)


In [16]:
gdf_predict = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/nisjedata-substrat-xgbclassifier_norge_latest_25833.geo.parquet")

In [22]:
# Ensure both layers share the same CRS (reproject points to match polygons)
gdf_blotbunn_proj = gdf_blotbunn.to_crs(gdf_predict.crs)

# Filter polygons with desired BunnType
total_points = len(gdf_blotbunn_proj)
for bunn_type in gdf_predict["BunnType"].unique():
    soft_bottom_polygons = gdf_predict[gdf_predict["BunnType"] == bunn_type]

    # Spatial join: keep only points that fall inside a "løsbunn" polygon
    points_in_soft_bottom = gpd.sjoin(
        gdf_blotbunn_proj,
        soft_bottom_polygons[["BunnType", "geometry"]],
        how="inner",
        predicate="within",
    )   

    # Count how many points are inside bløtbunn polygons
    n_points_in_soft_bottom = len(points_in_soft_bottom)
    
    percent_in_soft_bottom = (n_points_in_soft_bottom / total_points) * 100

    print(f"Points in {bunn_type}: {n_points_in_soft_bottom}")
    print(f"Percentage in {bunn_type}: {percent_in_soft_bottom:.2f}% /n")

Points in løsbunn: 2068
Percentage in løsbunn: 86.17% /n
Points in fastbunn: 148
Percentage in fastbunn: 6.17% /n
Points in blanding: 37
Percentage in blanding: 1.54% /n
